# Train cube landmarks and sticker colors with RF-DETR in Colab

This notebook fine-tunes **RF-DETR Keypoint Preview** using the same trainer as the local project. Run the cells in order. Choose **Runtime → Change runtime type → GPU** before starting.

Upload one ZIP with one or more generated dataset folders. An enclosing folder is fine. For example:

```text
dataset_a/images/000000.png
dataset_a/pose/000000.txt
dataset_a/sticker_keypoint_manifest.json
dataset_b/images/000000.png
dataset_b/pose/000000.txt
dataset_b/sticker_keypoint_manifest.json
```

Each dataset must use the same class names in the same class-ID order; its manifest preserves that order. Legacy three-class datasets with `independent_keypoint_manifest.json` are supported too. Put dataset folders in the ZIP directly, rather than ZIP files inside the ZIP. ZIPs may also contain other generator files. An optional second ZIP can contain cube-free negative images.

**Where results go:** Colab runtime storage is temporary. Set `MOUNT_DRIVE` and `SAVE_RESULTS_TO_DRIVE` below to write training checkpoints directly to Google Drive, or download the checkpoint from the final cell before the runtime ends. Drive mounting requires your authorization. GPU availability and session duration vary by Colab runtime.


## 1. Configure the run

Set the ZIP paths if the files already exist on Google Drive. Leave `DATASET_ZIP` blank to upload it from your computer when prompted. For optional negative images, set `NEGATIVE_ZIP` or enable `UPLOAD_NEGATIVES`. Paths may also point to files already in the Colab runtime.


In [ ]:
from pathlib import Path

#@title Inputs and output
MOUNT_DRIVE = False  #@param {type:"boolean"}
DATASET_ZIP = ""  #@param {type:"string"}
NEGATIVE_ZIP = ""  #@param {type:"string"}
UPLOAD_NEGATIVES = False  #@param {type:"boolean"}
SAVE_RESULTS_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/rfdetr_cube_runs"  #@param {type:"string"}

#@title Training
EPOCHS = 100  #@param {type:"integer"}
IMAGE_SIZE = 288  #@param {type:"integer"}
BATCH_SIZE = 4  #@param {type:"integer"}
GRAD_ACCUM_STEPS = 4  #@param {type:"integer"}
LEARNING_RATE = 2e-5  #@param {type:"number"}
ENCODER_LEARNING_RATE = 2e-5  #@param {type:"number"}
VALIDATION_FRACTION = 0.2  #@param {type:"number"}
SEED = 17  #@param {type:"integer"}
WORKERS = 0  #@param {type:"integer"}
PATIENCE = 25  #@param {type:"integer"}
USE_EMA = True  #@param {type:"boolean"}

if SAVE_RESULTS_TO_DRIVE and not MOUNT_DRIVE:
    raise ValueError("Set MOUNT_DRIVE = True to save results to Google Drive.")
if not MOUNT_DRIVE and any(str(p).startswith('/content/drive/') for p in (DATASET_ZIP, NEGATIVE_ZIP) if p):
    raise ValueError("Set MOUNT_DRIVE = True to use ZIPs stored in Google Drive.")
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

print('Configuration ready. Input resolution:', IMAGE_SIZE)


## 2. Check the GPU and install RF-DETR

This installs RF-DETR directly, without installing the project package or `kociemba`. The runtime must have a CUDA-enabled GPU.


In [ ]:
import subprocess
import sys
import shutil

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('No NVIDIA GPU found. Select a GPU runtime and reconnect.')
result = subprocess.run(['nvidia-smi'], check=False)
if result.returncode != 0:
    raise RuntimeError('No NVIDIA GPU found. Select a GPU runtime and reconnect.')
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'rfdetr[train]>=1.11.2,<2', 'PyYAML>=6.0',
], check=True)


In [ ]:
import torch
from importlib.metadata import version

print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('RF-DETR:', version('rfdetr'))
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot access a CUDA GPU. Check the runtime type and installation.')
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} ({props.total_memory / 1024**3:.1f} GiB VRAM)')
print('RF-DETR will print the actual training precision at startup.')


## 3. Provide your ZIPs

Use a ZIP containing only negative examples for the optional negatives input. Nested image folders are accepted. Large uploads work best from a Drive ZIP: the notebook extracts and prepares data on the Colab VM to avoid repeated small reads from Drive.


In [ ]:
import stat
import tempfile
import zipfile

SESSION_DIR = Path(tempfile.mkdtemp(prefix='rfdetr_cube_', dir='/content'))
INPUT_DIR = SESSION_DIR / 'inputs'
INPUT_DIR.mkdir()
print('Temporary session directory:', SESSION_DIR, flush=True)


def get_zip(configured_path, prompt):
    if configured_path:
        path = Path(configured_path).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f'ZIP not found: {path}')
        if path.suffix.lower() != '.zip':
            raise ValueError(f'Expected a .zip file: {path}')
        print(f'{prompt}: {path}', flush=True)
        return path
    from google.colab import files
    print(prompt, flush=True)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Select exactly one ZIP file.')
    name, contents = next(iter(uploaded.items()))
    if Path(name).suffix.lower() != '.zip':
        raise ValueError('The selected file must be a .zip archive.')
    path = INPUT_DIR / Path(name).name
    path.write_bytes(contents)
    return path


def extract_zip(path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(path) as archive:
        members = archive.infolist()
        for item in members:
            target = (root / item.filename).resolve()
            if not target.is_relative_to(root) or stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError(f'Unsafe ZIP entry: {item.filename}')
        print(f'Extracting {path.name} ({len(members):,} entries)...', flush=True)
        archive.extractall(root)


def find_dataset_roots(root):
    candidates = sorted(p for p in [root, *root.rglob('*')]
                  if p.is_dir() and '__MACOSX' not in p.parts
                  and (p / 'images').is_dir() and (p / 'pose').is_dir())
    if not candidates:
        raise ValueError('No dataset folders containing images/ and pose/ were found in the ZIP.')
    return candidates


dataset_zip = get_zip(DATASET_ZIP, 'Select your generated dataset ZIP')
dataset_extract = INPUT_DIR / 'dataset'
extract_zip(dataset_zip, dataset_extract)
SOURCE_DIRS = find_dataset_roots(dataset_extract)
print(f'Found {len(SOURCE_DIRS):,} dataset folder(s):', flush=True)
for source in SOURCE_DIRS:
    print(' ', source, flush=True)

NEGATIVES_DIR = None
if NEGATIVE_ZIP or UPLOAD_NEGATIVES:
    negative_zip = get_zip(NEGATIVE_ZIP, 'Select your negative-images ZIP')
    negative_extract = INPUT_DIR / 'negative_archive'
    extract_zip(negative_zip, negative_extract)
    negative_images = sorted(p for p in negative_extract.rglob('*')
        if p.is_file() and '__MACOSX' not in p.parts
        and p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'})
    if not negative_images:
        raise ValueError('No images found in the negative ZIP.')
    # The trainer expects a single flat directory. Copy with unique names.
    import shutil
    NEGATIVES_DIR = INPUT_DIR / 'negatives'
    NEGATIVES_DIR.mkdir()
    for index, image in enumerate(negative_images):
        shutil.copy2(image, NEGATIVES_DIR / f'negative_{index:07d}{image.suffix.lower()}')
    print(f'Prepared {len(negative_images):,} negative images.', flush=True)


## 4. Load the bundled trainer

The next cell writes a snapshot of the project trainer into the temporary runtime. It does not depend on a GitHub checkout.


In [ ]:
TRAINER_SOURCE = '"""Fine-tune RF-DETR Keypoint Preview on independent cube/sticker points.\n\nSources contain images/ and pose/ with YOLO rows:\n    class cx cy width height keypoint_x keypoint_y visibility\n\nClass order comes from the generator\'s sticker_keypoint_manifest.json or\nindependent_keypoint_manifest.json, then data.yaml, or --class-names.\n--extra-negative-dir retains the YOLO trainer\'s background-image interface.\nDataset preparation works without RF-DETR installed; training needs .[rfdetr].\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport random\nimport shutil\nfrom collections import Counter\nfrom pathlib import Path\n\n\nDEFAULT_CLASS_NAMES = ["corner", "corner-edge", "corner-edge-center"]\nIMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}\nKEYPOINT_SHAPE = [1, 3]\n\n\ndef parse_class_names(value: str) -> list[str]:\n    names = [name.strip() for name in value.split(",")]\n    if not all(names) or len(set(names)) != len(names):\n        raise argparse.ArgumentTypeError("Class names must be nonempty and unique.")\n    return names\n\n\ndef source_class_names(source_dir: Path) -> list[str]:\n    """Prefer pose manifests: legacy data.yaml names describe segmentation."""\n    import yaml\n\n    for filename in ("sticker_keypoint_manifest.json", "independent_keypoint_manifest.json"):\n        path = source_dir / filename\n        if path.is_file():\n            metadata = json.loads(path.read_text(encoding="utf-8"))\n            if metadata.get("keypoint_shape") != KEYPOINT_SHAPE:\n                raise ValueError(f"{path}: expected keypoint_shape {KEYPOINT_SHAPE}")\n            return parse_class_names(",".join(metadata["class_names"]))\n    path = source_dir / "data.yaml"\n    if path.is_file():\n        metadata = yaml.safe_load(path.read_text(encoding="utf-8"))\n        if metadata.get("kpt_shape") != KEYPOINT_SHAPE:\n            raise ValueError(f"{path}: expected kpt_shape {KEYPOINT_SHAPE}")\n        names = metadata["names"]\n        if isinstance(names, dict):\n            indexed_names = {int(key): value for key, value in names.items()}\n            if sorted(indexed_names) != list(range(len(indexed_names))):\n                raise ValueError(f"{path}: class IDs must be contiguous, starting at zero")\n            names = [indexed_names[index] for index in range(len(indexed_names))]\n        return parse_class_names(",".join(names))\n    return DEFAULT_CLASS_NAMES.copy()\n\n\ndef resolve_class_names(source_dirs: list[Path], override: list[str] | None) -> list[str]:\n    if override is not None:\n        return override\n    names = source_class_names(source_dirs[0])\n    for source_dir in source_dirs[1:]:\n        if source_class_names(source_dir) != names:\n            raise ValueError(\n                f"Class order differs between sources: {source_dirs[0]} and {source_dir}. "\n                "Remap labels before combining datasets."\n            )\n    return names\n\n\ndef validate_label_file(path: Path, class_names: list[str]) -> Counter[int]:\n    counts: Counter[int] = Counter()\n    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):\n        if not line.strip():\n            continue\n        prefix = f"{path}:{line_number}"\n        try:\n            values = [float(part) for part in line.split()]\n        except ValueError as exc:\n            raise ValueError(f"{prefix}: nonnumeric pose label") from exc\n        if len(values) != 8 or not all(math.isfinite(value) for value in values):\n            raise ValueError(f"{prefix}: expected 8 finite fields for a one-keypoint pose")\n        class_id, cx, cy, width, height, x, y, visibility = values\n        if not class_id.is_integer() or not 0 <= class_id < len(class_names):\n            raise ValueError(f"{prefix}: invalid class ID {class_id}")\n        if not all(0 <= value <= 1 for value in (cx, cy, width, height, x, y)):\n            raise ValueError(f"{prefix}: box/keypoint coordinates must be normalized to [0, 1]")\n        if width <= 0 or height <= 0:\n            raise ValueError(f"{prefix}: bounding-box width and height must be positive")\n        if visibility not in (0, 1, 2):\n            raise ValueError(f"{prefix}: visibility must be 0, 1, or 2")\n        counts[int(class_id)] += 1\n    return counts\n\n\ndef write_dataset(\n    source_dirs: list[Path],\n    dataset_dir: Path,\n    val_fraction: float,\n    seed: int,\n    pose_subdir: str = "pose",\n    extra_negative_dir: Path | None = None,\n    class_names: list[str] | None = None,\n    skip_missing_labels: bool = False,\n) -> Path:\n    """Validate all labels, then prepare reproducible train/val splits."""\n    import yaml\n\n    if not source_dirs or not 0 <= val_fraction < 1:\n        raise ValueError("Supply sources and a validation fraction in [0, 1).")\n    source_dirs = list(dict.fromkeys(path.resolve() for path in source_dirs))\n    class_names = resolve_class_names(source_dirs, class_names)\n    dataset_dir = dataset_dir.resolve()\n    inputs = source_dirs + ([extra_negative_dir.resolve()] if extra_negative_dir else [])\n    for input_dir in inputs:\n        if (\n            dataset_dir == input_dir\n            or dataset_dir in input_dir.parents\n            or input_dir in dataset_dir.parents\n        ):\n            raise ValueError(f"Prepared dataset must be separate from input directory: {input_dir}")\n    if dataset_dir.exists() and not (\n        (dataset_dir / "conversion_manifest.json").is_file()\n        and (dataset_dir / "data.yaml").is_file()\n    ):\n        raise FileExistsError(\n            f"Refusing to replace an unrecognized dataset directory: {dataset_dir}"\n        )\n\n    samples = []\n    seen_images: set[Path] = set()\n    skipped_missing = 0\n    for source_dir in source_dirs:\n        images_dir, labels_dir = source_dir / "images", source_dir / pose_subdir\n        if not images_dir.is_dir() or not labels_dir.is_dir():\n            raise FileNotFoundError(f"Expected {images_dir} and {labels_dir}")\n        images = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)\n        if not images:\n            raise ValueError(f"No images found: {images_dir}")\n        print(f"Validating {len(images):,} pose labels from {source_dir}", flush=True)\n        for image_index, image_path in enumerate(images, 1):\n            label_path = labels_dir / f"{image_path.stem}.txt"\n            if not label_path.is_file():\n                if not skip_missing_labels:\n                    raise FileNotFoundError(f"Missing pose label: {label_path}")\n                skipped_missing += 1\n                print(f"  Skipping image with missing pose label: {image_path}", flush=True)\n                continue\n            counts = validate_label_file(label_path, class_names)\n            if image_path.resolve() not in seen_images:\n                samples.append((image_path, label_path, counts))\n                seen_images.add(image_path.resolve())\n            if image_index % 1000 == 0 or image_index == len(images):\n                print(f"  Validated {image_index:,}/{len(images):,} labels", flush=True)\n    if skipped_missing:\n        print(f"Skipped {skipped_missing:,} image(s) with missing pose labels.", flush=True)\n    if not samples:\n        raise ValueError("No images with pose labels found in the dataset sources.")\n    if extra_negative_dir is not None:\n        if not extra_negative_dir.is_dir():\n            raise FileNotFoundError(f"Missing extra-negative image directory: {extra_negative_dir}")\n        print(f"Collecting negative images from {extra_negative_dir}", flush=True)\n        for image_path in sorted(extra_negative_dir.iterdir()):\n            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:\n                if image_path.resolve() in seen_images:\n                    raise ValueError(\n                        f"Image supplied as both labeled and extra-negative: {image_path}"\n                    )\n                samples.append((image_path.resolve(), None, Counter()))\n\n    # Split foreground and backgrounds separately so negatives cannot displace\n    # all labeled validation images in a small dataset.\n    rng = random.Random(seed)\n    splits: dict[str, list] = {"train": [], "val": []}\n    for group in (\n        [sample for sample in samples if sample[2]],\n        [sample for sample in samples if not sample[2]],\n    ):\n        rng.shuffle(group)\n        val_count = (\n            min(len(group) - 1, max(1, round(len(group) * val_fraction)))\n            if len(group) > 1 and val_fraction > 0\n            else 0\n        )\n        splits["val"].extend(group[:val_count])\n        splits["train"].extend(group[val_count:])\n\n    # No output is replaced until every source label and directory is valid.\n    if dataset_dir.exists():\n        print(f"Replacing prepared dataset: {dataset_dir}", flush=True)\n        shutil.rmtree(dataset_dir)\n    manifest, summary = [], {}\n    for split, split_samples in splits.items():\n        rng.shuffle(split_samples)\n        images_dir, labels_dir = dataset_dir / split / "images", dataset_dir / split / "labels"\n        images_dir.mkdir(parents=True)\n        labels_dir.mkdir(parents=True)\n        class_counts: Counter[int] = Counter()\n        print(f"Preparing {split} split: {len(split_samples):,} images", flush=True)\n        for index, (image_path, label_path, counts) in enumerate(split_samples):\n            stem = f"{split}_{index:06d}"\n            target_image = images_dir / f"{stem}{image_path.suffix.lower()}"\n            target_label = labels_dir / f"{stem}.txt"\n            shutil.copy2(image_path, target_image)\n            if label_path is None:\n                target_label.write_text("", encoding="utf-8")\n            else:\n                shutil.copy2(label_path, target_label)\n            class_counts.update(counts)\n            manifest.append(\n                {\n                    "split": split,\n                    "source_image": str(image_path),\n                    "source_label": str(label_path) if label_path else None,\n                    "image": str(target_image),\n                    "label": str(target_label),\n                    "objects": sum(counts.values()),\n                    "negative": not bool(counts),\n                }\n            )\n            copied = index + 1\n            if copied % 1000 == 0 or copied == len(split_samples):\n                print(f"  Copied {copied:,}/{len(split_samples):,} {split} images", flush=True)\n        summary[split] = {\n            "images": len(split_samples),\n            "negative_images": sum(not bool(sample[2]) for sample in split_samples),\n            "class_objects": {name: class_counts[index] for index, name in enumerate(class_names)},\n        }\n    (dataset_dir / "conversion_manifest.json").write_text(\n        json.dumps(manifest, indent=2) + "\\n", encoding="utf-8"\n    )\n    (dataset_dir / "dataset_summary.json").write_text(\n        json.dumps(summary, indent=2) + "\\n", encoding="utf-8"\n    )\n    data_yaml = dataset_dir / "data.yaml"\n    data_yaml.write_text(\n        yaml.safe_dump(\n            {\n                "path": str(dataset_dir),\n                "train": "train/images",\n                "val": "val/images",\n                "names": dict(enumerate(class_names)),\n                "kpt_shape": KEYPOINT_SHAPE,\n            },\n            sort_keys=False,\n        ),\n        encoding="utf-8",\n    )\n    print("Dataset summary:")\n    print(json.dumps(summary, indent=2), flush=True)\n    return data_yaml\n\n\ndef train(data_yaml: Path, runs_dir: Path, args: argparse.Namespace) -> Path:\n    import yaml\n\n    summary = json.loads((data_yaml.parent / "dataset_summary.json").read_text())\n    if any(not sum(summary[split]["class_objects"].values()) for split in ("train", "val")):\n        raise ValueError(\n            "Training needs labeled images in both splits; use at least two positives."\n        )\n    try:\n        from rfdetr import RFDETRKeypointPreview\n    except ImportError as exc:\n        raise RuntimeError(\n            \'RF-DETR Keypoint training requires: python -m pip install -e ".[rfdetr]"\'\n        ) from exc\n\n    class_names = list(yaml.safe_load(data_yaml.read_text())["names"].values())\n    model_kwargs = {\n        "num_classes": len(class_names),\n        "num_keypoints_per_class": [1] * len(class_names),\n        "resolution": args.imgsz,\n    }\n    if args.model is not None:\n        model_kwargs["pretrain_weights"] = str(args.model)\n    if args.device is not None:\n        model_kwargs["device"] = args.device\n    print("Initializing RF-DETR Keypoint Preview (may download pretrained weights)...", flush=True)\n    model = RFDETRKeypointPreview(**model_kwargs)\n\n    output_dir = runs_dir / args.name\n    suffix = 2\n    while output_dir.exists():\n        output_dir = runs_dir / f"{args.name}-{suffix}"\n        suffix += 1\n    train_kwargs = {\n        "dataset_dir": str(data_yaml.parent),\n        "dataset_file": "yolo",\n        "class_names": class_names,\n        "output_dir": str(output_dir.resolve()),\n        "epochs": args.epochs,\n        "batch_size": args.batch,\n        "grad_accum_steps": args.grad_accum_steps,\n        "lr": args.lr,\n        "lr_encoder": args.lr_encoder,\n        "num_workers": args.workers,\n        "seed": args.seed,\n        "early_stopping": args.patience > 0,\n        "early_stopping_patience": args.patience,\n        "skip_best_epochs": min(10, args.epochs - 1),\n        "run_test": False,\n        "use_ema": args.ema,\n        "tensorboard": args.tensorboard,\n        "progress_bar": "tqdm",\n        # Synthetic data already supplies camera/lighting diversity. Keep the\n        # fixed input size and physical handedness; don\'t shift hue/color labels.\n        "multi_scale": False,\n        "expanded_scales": False,\n        "scale_jitter": False,\n        "aug_config": {},\n        "keypoint_flip_pairs": [],\n    }\n    if args.device is not None:\n        train_kwargs["device"] = args.device\n    print(\n        f"Starting training: {args.epochs} epochs, {args.imgsz}x{args.imgsz}, "\n        f"batch {args.batch}, device {args.device or \'auto\'}\\n"\n        f"Output directory: {output_dir.resolve()}",\n        flush=True,\n    )\n    model.train(**train_kwargs)\n    best_model_path = output_dir / "checkpoint_best_total.pth"\n    if not best_model_path.is_file():\n        raise FileNotFoundError(f"Training completed without a best checkpoint: {best_model_path}")\n    print(f"Best model: {best_model_path.resolve()}")\n    return best_model_path.resolve()\n\n\ndef parse_args(argv: list[str] | None = None) -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\n        "--source-dir",\n        "--source-dirs",\n        dest="source_dir_groups",\n        type=Path,\n        nargs="+",\n        action="append",\n        help="Dataset root(s); repeat to combine sources.",\n    )\n    parser.add_argument("--pose-subdir", default="pose")\n    parser.add_argument(\n        "--extra-negative-dir",\n        type=Path,\n        default=None,\n        help="Cube-free images to include with empty pose labels (omitted by default).",\n    )\n    parser.add_argument(\n        "--class-names",\n        type=parse_class_names,\n        help="Comma-separated class names in label ID order; overrides source metadata.",\n    )\n    parser.add_argument("--workdir", type=Path, default=Path("runs/rfdetr_keypoints_local"))\n    parser.add_argument("--val-fraction", type=float, default=0.2)\n    parser.add_argument("--seed", type=int, default=17)\n    parser.add_argument("--model", type=Path, help="Optional pretrained checkpoint override.")\n    parser.add_argument("--epochs", type=int, default=100)\n    parser.add_argument("--imgsz", "--resolution", type=int, default=288)\n    parser.add_argument("--batch", type=int, default=4)\n    parser.add_argument("--grad-accum-steps", type=int, default=4)\n    parser.add_argument("--lr", type=float, default=2e-5)\n    parser.add_argument("--lr-encoder", type=float, default=2e-5)\n    parser.add_argument("--workers", type=int, default=0)\n    parser.add_argument("--device", help="cpu, mps, cuda, cuda:N, or a GPU index such as 0.")\n    parser.add_argument("--name", default="cube_keypoints_rfdetr")\n    parser.add_argument("--patience", type=int, default=25, help="Early-stop patience; 0 disables.")\n    parser.add_argument("--ema", action=argparse.BooleanOptionalAction, default=True)\n    parser.add_argument("--tensorboard", action=argparse.BooleanOptionalAction, default=False)\n    parser.add_argument("--prepare-only", action="store_true")\n    args = parser.parse_args(argv)\n    if args.imgsz <= 0 or args.imgsz % 24:\n        parser.error("--imgsz must be a positive multiple of 24 (e.g. 144, 264, 288).")\n    if not 0 <= args.val_fraction < 1:\n        parser.error("--val-fraction must be in [0, 1).")\n    if min(args.epochs, args.batch, args.grad_accum_steps) <= 0:\n        parser.error("--epochs, --batch, and --grad-accum-steps must be positive.")\n    if args.workers < 0 or args.patience < 0:\n        parser.error("--workers and --patience must be nonnegative.")\n    if not all(math.isfinite(rate) and rate > 0 for rate in (args.lr, args.lr_encoder)):\n        parser.error("Learning rates must be positive and finite.")\n    if not args.name or Path(args.name).name != args.name or args.name in (".", ".."):\n        parser.error("--name must be a single directory name.")\n    if args.device is not None:\n        args.device = f"cuda:{args.device}" if args.device.isdigit() else args.device\n        if args.device not in ("cpu", "mps", "cuda") and not (\n            args.device.startswith("cuda:") and args.device[5:].isdigit()\n        ):\n            parser.error("--device must be cpu, mps, cuda, cuda:N, or a GPU index.")\n    return args\n\n\ndef main() -> None:\n    args = parse_args()\n    source_dirs = [\n        path\n        for group in (args.source_dir_groups or [[Path("data/synthetic_cube_dataset")]])\n        for path in group\n    ]\n    data_yaml = write_dataset(\n        source_dirs,\n        args.workdir / "dataset",\n        args.val_fraction,\n        args.seed,\n        args.pose_subdir,\n        args.extra_negative_dir,\n        args.class_names,\n    )\n    if args.prepare_only:\n        print(f"Prepared dataset: {data_yaml.parent}")\n        return\n    train(data_yaml, args.workdir / "runs", args)\n\n\nif __name__ == "__main__":\n    main()\n'
TRAINER_PATH = SESSION_DIR / 'train_rfdetr_keypoints_local.py'
TRAINER_PATH.write_text(TRAINER_SOURCE, encoding='utf-8')

import importlib.util
spec = importlib.util.spec_from_file_location('cube_rfdetr_trainer', TRAINER_PATH)
trainer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(trainer)
print('Loaded trainer:', TRAINER_PATH)


## 5. Validate and prepare the dataset

The trainer validates each pose label and prints progress. Images missing a corresponding pose label are skipped and reported; they are not treated as negative examples. Images and empty negative labels are copied to the Colab runtime; train and validation splits are deterministic for the selected seed.


In [ ]:
PREPARED_DIR = SESSION_DIR / 'dataset'
DATA_YAML = trainer.write_dataset(
    source_dirs=SOURCE_DIRS,
    dataset_dir=PREPARED_DIR,
    val_fraction=VALIDATION_FRACTION,
    seed=SEED,
    pose_subdir='pose',
    extra_negative_dir=NEGATIVES_DIR,
    skip_missing_labels=True,
)
print('Prepared dataset:', DATA_YAML)


## 6. Train

Training displays a live progress bar and validation metrics. The model downloads its pretrained checkpoint on first use. If you hit CUDA out-of-memory, lower `BATCH_SIZE` and rerun from the configuration cell. Your run directory is unique, so previous checkpoints are preserved.


In [ ]:
args_list = [
    '--epochs', str(EPOCHS), '--imgsz', str(IMAGE_SIZE),
    '--batch', str(BATCH_SIZE), '--grad-accum-steps', str(GRAD_ACCUM_STEPS),
    '--lr', str(LEARNING_RATE), '--lr-encoder', str(ENCODER_LEARNING_RATE),
    '--val-fraction', str(VALIDATION_FRACTION), '--seed', str(SEED),
    '--workers', str(WORKERS), '--patience', str(PATIENCE), '--device', 'cuda',
]
if not USE_EMA:
    args_list.append('--no-ema')
args = trainer.parse_args(args_list)
RUNS_DIR = Path(DRIVE_RESULTS_DIR).expanduser() if SAVE_RESULTS_TO_DRIVE else SESSION_DIR / 'runs'
print('Training output root:', RUNS_DIR, flush=True)
BEST_CHECKPOINT = trainer.train(DATA_YAML, RUNS_DIR, args)
print('Best checkpoint:', BEST_CHECKPOINT, flush=True)


## 7. Download or locate your checkpoint

This cell downloads the best checkpoint to your computer. If Google Drive output is enabled, the full run directory, including checkpoints and training configuration, is already on Drive.


In [ ]:
if 'BEST_CHECKPOINT' not in globals() or not Path(BEST_CHECKPOINT).is_file():
    raise FileNotFoundError('No completed best checkpoint. Finish training first, or inspect the run directory.')
print('Best checkpoint:', BEST_CHECKPOINT)
if SAVE_RESULTS_TO_DRIVE:
    print('Saved on Google Drive:', BEST_CHECKPOINT)
from google.colab import files
files.download(str(BEST_CHECKPOINT))
